# TLA-Prover evidence lab

This notebook keeps three claims separate: TLAKit works, a cluster is reachable, and a prover candidate improves the frozen gate. Only the last claim can promote a model.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'tools').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import tlakit
from tlakit.remote import RemoteRunner
from tools.tlakit_cluster_backend import default_backends

print('TLAKit', tlakit.__version__)

## Ground-truth toolchain smoke check

This exercises TLAKit and SANY through the public runner. It is a toolchain check, not evidence that the learned prover improved.

In [ ]:
SOURCE = r'''---- MODULE NotebookSmoke ----
EXTENDS Naturals
THEOREM OnePlusOne == 1 + 1 = 2
===='''
runner = RemoteRunner()
health = runner.health()
parsed = runner.parse(SOURCE, 'NotebookSmoke')
{'service': health, 'outcome': parsed.outcome.value, 'diagnostics': len(parsed.diagnostics)}

## Polaris and Sophia

The probe is read-only and non-interactive. `endpoint_healthy=None` means no private HTTP runner was configured; failed SSH authentication is reported rather than prompting inside the notebook.

In [ ]:
cluster_status = [asdict(backend.probe()) for backend in default_backends()]
cluster_status

## SkillOpt-derived experiment discipline

We borrow the mechanics, not the metric, from [Microsoft SkillOpt](https://github.com/microsoft/SkillOpt): preserve scored trajectories, propose bounded changes, keep rejected changes as negative evidence, and accept only strict held-out improvement. The protected 119-case gate remains fixed and never becomes training data.

In [ ]:
experiment_policy = {
    'target': 'frozen prover checkpoint',
    'optimizer_input': 'non-protected scored trajectories only',
    'update_budget': 'one bounded hypothesis per child artifact',
    'negative_memory': 'retain rejected edits and failure signatures',
    'promotion': 'strict protected-gate improvement with SANY/proof evidence',
    'forbidden_proxies': ['training loss', 'SSH health', 'TLAKit health', 'optimizer preference'],
}
experiment_policy